# Kumo Relational Quickstart with structured-data-models-client

**Kumo Relational** is a relational foundation model for
machine learning on enterprise data. With your relational data and a
Predictive Query Language (PQL) query, you can generate predictions without
training a task-specific model.

This quickstart uses four tables from Microsoft's **AdventureWorks** OLTP
sample database: customers, products, sales-order headers, and sales-order
details. Predictions are submitted to a deployed Kumo Relational NVIDIA NIM through
`structured-data-models-client`.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.

## Introduction

Kumo Relational is grounded in three key world views:

> **1. Enterprise data is a graph.**

Enterprise data is a graph where tables are connected by keys. In this notebook, `sales_order_headers` connects customers to orders, while `sales_order_details` connects each order to its products. Together, these relationships give Kumo Relational the business context surrounding every transaction.

Once we structure enterprise data this way, we can apply pre-trained Relational Graph Transformers to extract insights and patterns.

> **2. With timestamps, we place events on a timeline.**

By placing events on a timeline, we unlock the ability to model how things evolve. We can select any point in time and predict what is likely to happen next from the sequence and patterns in the historical data.

> **3. Machine learning tasks can be described via predictive queries.**

Major machine learning tasks, including regression, classification, forecasting, and recommendation, can be expressed using *Predictive Query Language (PQL)*. A PQL query identifies the outcome to predict, the entity to predict it for, and, for temporal tasks, the future prediction window.

If you know SQL, PQL should feel familiar. Learn more in the [query-writing guide](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/writing-predictive-queries).

**Let's get started!**

## Step 1. Install structured-data-models-client

Install the SDK with the relational extra. See [Deploy, Install, and Connect](https://docs.nvidia.com/sdgm/kumo-relational/get-started/sdk-getting-started) for the supported Python versions and platforms.

In [ ]:
%pip install "structured-data-models-client[relational]"

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)

Build schemas and queries through `sdm_client`'s top-level names: `Schema`, `Relationship`, and `PredictiveQuery`. Submit predictions through `SDMClient`, naming the model with `KumoRelational()`.

## Step 2. Configure the Kumo Relational NIM endpoint

Deploy Kumo Relational as an NVIDIA NIM before running this notebook. See
[Deploy, Install, and Connect](https://docs.nvidia.com/sdgm/kumo-relational/get-started/sdk-getting-started)
for deployment instructions.

Set `KUMO_RELATIONAL_API_ENDPOINT` to the reachable NIM endpoint. The default below uses a
local container on port `8000`. Set `KUMO_RELATIONAL_API_KEY` only when an
authenticating gateway protects the endpoint.

In [ ]:
NIM_URL = os.environ.get(
    'KUMO_RELATIONAL_API_ENDPOINT',
    'http://<dns_name>:8000',
)
NIM_API_KEY = os.environ.get('KUMO_RELATIONAL_API_KEY')

## Step 3. Connect to the NIM

An `SDMClient` owns the HTTP connection to one NIM endpoint. Keep credentials
in environment configuration rather than notebook source.

In [ ]:
client = SDMClient(
    url=NIM_URL,
    api_key=NIM_API_KEY,
)

print('Ready:', client.health_ready())
print('Capabilities:', client.capabilities(KumoRelational()))

<div class="alert alert-warning">
<strong>Security:</strong> Never hard-code NIM or gateway credentials in a
notebook. Use environment variables or your platform's secret manager.
</div>

## Step 4. Data Preparation

This notebook uses the **AdventureWorks OLTP sample database**, originally
published by Microsoft as a SQL Server sample. The four source extracts are:

- `Customer.csv`
- `Product.csv`
- `SalesOrderHeader.csv`
- `SalesOrderDetail.csv`

Download the four `*.csv` extracts from the AdventureWorks OLTP install scripts in the
[Microsoft SQL Server samples repository](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works/oltp-install-script).

> Each user is responsible for checking the content of datasets and the
> applicable licenses and determining if suitable for the intended use.
The repository's samples and templates are provided under the
[MIT License](https://github.com/microsoft/sql-server-samples/blob/master/license.txt).

Set `DATA_LOCATION` to the Snowflake database and schema that hold the
AdventureWorks tables (`RUTH.ADVENTUREWORKS` in this account). Each table is
read with the Snowpark session, and only the listed columns that exist in the
table are kept.

The sales-order header and detail tables remain separate. During preparation,
only `CustomerID` and `OrderDate` are copied from each header to its detail
rows using `SalesOrderID`.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATA_LOCATION = '<database>.<schema>'
print('AdventureWorks data:', DATA_LOCATION)

CUSTOMER_COLUMNS = [
    'CustomerID',
    'PersonID',
    'StoreID',
    'TerritoryID',
    'AccountNumber',
    'rowguid',
    'ModifiedDate',
]

PRODUCT_COLUMNS = [
    'ProductID',
    'Name',
    'ProductNumber',
    'MakeFlag',
    'FinishedGoodsFlag',
    'Color',
    'SafetyStockLevel',
    'ReorderPoint',
    'StandardCost',
    'ListPrice',
    'Size',
    'SizeUnitMeasureCode',
    'WeightUnitMeasureCode',
    'Weight',
    'DaysToManufacture',
    'ProductLine',
    'Class',
    'Style',
    'ProductSubcategoryID',
    'ProductModelID',
    'SellStartDate',
    'SellEndDate',
    'DiscontinuedDate',
    'rowguid',
    'ModifiedDate',
]

SALES_HEADER_COLUMNS = [
    'SalesOrderID',
    'RevisionNumber',
    'OrderDate',
    'DueDate',
    'ShipDate',
    'Status',
    'OnlineOrderFlag',
    'SalesOrderNumber',
    'PurchaseOrderNumber',
    'AccountNumber',
    'CustomerID',
    'SalesPersonID',
    'TerritoryID',
    'BillToAddressID',
    'ShipToAddressID',
    'ShipMethodID',
    'CreditCardID',
    'CreditCardApprovalCode',
    'CurrencyRateID',
    'SubTotal',
    'TaxAmt',
    'Freight',
    'TotalDue',
    'Comment',
    'rowguid',
    'ModifiedDate',
]

SALES_DETAIL_COLUMNS = [
    'SalesOrderID',
    'SalesOrderDetailID',
    'CarrierTrackingNumber',
    'OrderQty',
    'ProductID',
    'SpecialOfferID',
    'UnitPrice',
    'UnitPriceDiscount',
    'LineTotal',
    'rowguid',
    'ModifiedDate',
]


def read_headerless_table(filename, columns):
    path = f'{DATA_LOCATION}."{Path(filename).stem}"'
    return session.table(path).to_pandas()


customers_df = read_headerless_table('Customer.csv', CUSTOMER_COLUMNS)
products_df = read_headerless_table('Product.csv', PRODUCT_COLUMNS)
sales_order_headers_df = read_headerless_table(
    'SalesOrderHeader.csv', SALES_HEADER_COLUMNS
)
sales_order_details_df = read_headerless_table(
    'SalesOrderDetail.csv', SALES_DETAIL_COLUMNS
)

Parse the timestamp fields, then enrich each sales-order detail with only its parent order's `CustomerID` and `OrderDate`. The header and detail dataframes remain separate.

In [ ]:
timestamp_columns = {
    'customers': (customers_df, ['ModifiedDate']),
    'products': (
        products_df,
        ['SellStartDate', 'SellEndDate', 'DiscontinuedDate', 'ModifiedDate'],
    ),
    'sales_order_headers': (
        sales_order_headers_df,
        ['OrderDate', 'DueDate', 'ShipDate', 'ModifiedDate'],
    ),
    'sales_order_details': (sales_order_details_df, ['ModifiedDate']),
}

for _, (table_df, columns) in timestamp_columns.items():
    for column in columns:
        if (
            column in table_df
        ):  # skip columns not present in the Snowflake table
            table_df[column] = pd.to_datetime(table_df[column], errors='coerce')

sales_order_details_df = sales_order_details_df.merge(
    sales_order_headers_df[['SalesOrderID', 'CustomerID', 'OrderDate']],
    on='SalesOrderID',
    how='left',
    validate='many_to_one',
)

## Step 5. Build a schema from DataFrames

`Schema.from_data()` accepts a dictionary of pandas DataFrames and infers
each table's column types and semantic roles automatically. The AdventureWorks
relationships are supplied explicitly so the example is reproducible.
Kumo Relational still infers column data types and semantic metadata, which
you should review before prediction.

In [ ]:
schema = Schema.from_data(
    {
        'customers': customers_df,
        'products': products_df,
        'sales_order_headers': sales_order_headers_df,
        'sales_order_details': sales_order_details_df,
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_headers': 'SalesOrderID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        # These two tables contain datetime columns but do not represent
        # event timelines, so no time column is assigned to them.
        'customers': None,
        'products': None,
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
    },
    relationships=[
        Relationship(
            source_table='sales_order_headers',
            source_column='CustomerID',
            target_table='customers',
            target_column='CustomerID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='SalesOrderID',
            target_table='sales_order_headers',
            target_column='SalesOrderID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='ProductID',
            target_table='products',
            target_column='ProductID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='CustomerID',
            target_table='customers',
            target_column='CustomerID',
        ),
    ],
)

print(schema)
for name in schema.tables:
    print(schema[name])

Primary-key and relationship columns are automatically assigned the `ID`
semantic type. Review the remaining columns before prediction:

- Override a column's semantic type with
  `schema["table_name"]["column_name"].stype = "<type>"`.
- Available semantic types: `"numerical"`, `"categorical"`,
  `"multicategorical"`, `"ID"`, `"text"`, `"timestamp"`.

For this known AdventureWorks schema, review identifier-like strings before
prediction. Account numbers, product numbers, order numbers, approval codes,
`rowguid` columns, and `CarrierTrackingNumber` identify records rather than
categories, and several exceed the NIM's 10,000-value categorical cardinality
limit in the full extracts, so this example marks them `"ID"`. Product names
are typed `"text"` to reflect their content, though Kumo Relational currently
ignores `text` features for prediction; see
[Data Types](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/prepare-data/data-types).
Order quantity is a measured numerical value.

In [ ]:
# Correct semantic types that are specific to this known schema. These values
# identify business records rather than categories or free-text documents.
id_columns = {
    'customers': ['AccountNumber', 'rowguid'],
    'products': ['ProductNumber', 'rowguid'],
    'sales_order_headers': [
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
        'rowguid',
    ],
    'sales_order_details': ['CarrierTrackingNumber', 'rowguid'],
}

for table_name, column_names in id_columns.items():
    for column_name in column_names:
        if (
            column_name in schema[table_name].columns
        ):  # skip columns not in the Snowflake table
            schema[table_name][column_name].stype = 'ID'

# Product names carry natural-language meaning; order quantity is measured.
# This NIM does not read the 'text' stype, so Name is typed categorical.
schema['products']['Name'].stype = 'categorical'
schema['sales_order_details']['OrderQty'].stype = 'numerical'

Call `schema.check_consistent()` after all metadata changes. An incompatible
semantic type, a missing column, or a primary key that is also configured as
a time column raises an error.

In [ ]:
schema.check_consistent()
print('Schema OK.')
for name in schema.tables:
    print(schema[name])

## Step 6. Submit prediction requests

The schema is a reusable description of the data. For each prediction:

1. Write the task in PQL.
2. Build a `PredictiveQuery` from the query text and the schema.
3. Call `.materialize(indices=...)` to gather a `Request`.
4. Call `client.predict(request, model=KumoRelational())`.

The result's `to_pandas()` method returns a pandas DataFrame.

**Note:** AdventureWorks is sample data, and the queries and results below are intended for demonstration. Benchmark Kumo Relational on data representative of your own use case before drawing performance conclusions.

#### PQL: How you describe a prediction task

Learn more in the [query-writing guide](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/writing-predictive-queries).

The short reference below covers the entity, target, aggregation,
prediction-window, and condition syntax used by the examples in this
quickstart.

<details>
<summary><b>💡 Click here for a short introduction to Predictive Query!</b></summary>

**Entities** can be specified in three ways:

* `FOR <entity_table>.<entity_primary_key>=1` embeds one entity ID in PQL.
* `FOR <entity_table>.<entity_primary_key> IN (1, 2, 3)` embeds several
  entity IDs in PQL.
* `FOR EACH <entity_table>.<entity_primary_key>` declares the entity column,
  while `materialize(indices=[1, 2, 3])` supplies the entity IDs separately.

**Preferred:** For application code, use `FOR EACH` and pass primary-key values
through `materialize()`'s `indices` argument. This keeps the prediction task separate
from the entity batch and makes the same PQL reusable. Despite its name,
`indices` contains source primary-key values, not DataFrame row numbers.
If `indices` is provided, it overrides entity IDs embedded in the PQL.

**Targets** can be specified in two common ways:

* `PREDICT <target_table>.<target_column>` imputes a missing value.
* `PREDICT <aggregation>(<target>, <start_offset>, <end_offset>, <time_unit>)` predicts an aggregate in a future window.

Supported aggregations include `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, and `LIST_DISTINCT`. For example, `PREDICT COUNT(sales_order_headers.*, 0, 7, days)` predicts the number of orders in the next seven days.

Conditions can turn an aggregate into an event. For example, appending `=0` predicts the probability that no orders occur in the specified window.
</details>

### Example 1A: Forecast 30-day product demand

Predict how many units of product `ProductID=870` will be ordered in the next 30 days.

In [ ]:
query = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) FOR EACH products.ProductID',
    schema,
)

request = query.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the product with `ProductID=870`
2. `prediction`: the predicted number of units ordered during the next 30 days

This result can support inventory and demand planning.

### Example 1B: Forecast from a historical anchor time

By default, predictions use the maximum timestamp in the temporal schema. Set a historical `anchor_time` to simulate what the prediction would have looked like at that point in time. Kumo Relational uses only information available before the anchor time.

In [ ]:
# Pin the anchor instead of letting it be derived. It is taken from the
# data rather than written as a literal: AdventureWorks is re-dated
# upstream, so a hardcoded date stops matching the file you just read.
anchor_time = sales_order_headers_df['OrderDate'].max() - pd.Timedelta(days=60)

request = query.materialize(indices=[870], anchor_time=anchor_time)
result = client.predict(request, model=KumoRelational())
result.to_pandas()

### Example 1C: Forecast for multiple steps

In [ ]:
query_multi_step = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
    schema,
)

request = query_multi_step.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

A multi-step forecast returns one row per timeframe.
`forecast_step` is the one-based sequence number and `prediction` is the
forecast value for that period.

### Example 2: Predict customer inactivity

Predict the likelihood that customers `CustomerID=11176` and `CustomerID=11091` will place zero orders in the next 90 days.

In [ ]:
query_inactivity = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 FOR EACH customers.CustomerID',
    schema,
)

request = query_inactivity.materialize(indices=[11176, 11091])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the customer represented by each `CustomerID`
2. `prediction`: whether the zero-order event is predicted to occur
3. `probability_*` columns: the estimated likelihood of each outcome

This result can support retention or re-engagement workflows.

### Example 3: Recommend products

Rank which of a small candidate set of products customer `CustomerID=11176` is most likely to order in the next 30 days.

Ranking runs as one classification call per candidate, so the candidate set is
named explicitly with `candidates=` rather than left for the client to infer.
This example candidates the 15 most frequently ordered products so each has
enough purchase history for the model to learn from; an arbitrary or rarely
ordered candidate set can fail with "nothing to rank" because its context
holds only one outcome. See
[Prediction Types](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/prediction-types#recommendations-ranking).

In [ ]:
candidate_product_ids = (
    sales_order_details_df['ProductID']
    .value_counts()
    .head(15)
    .index.to_series()
    .reset_index(drop=True)
)

query_rank = PredictiveQuery(
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 10 FOR EACH customers.CustomerID',
    schema,
)

request = query_rank.materialize(
    indices=[11176],
    candidates=candidate_product_ids,
    num_context_samples=5000,
)
result = client.predict(request, model=KumoRelational())
print(result.to_pandas())
print('Unscored candidates:', result.unscored_candidates)

How to interpret the result:

1. `id`: the customer with `CustomerID=11176`
2. `rankings`: the ranked candidates, each with a `candidate` product ID and a `score`

A candidate whose sampled context holds only one outcome can't be scored and
is left out rather than failing the whole ranking; check
`result.unscored_candidates` for which ones were skipped.

### Example 4: Infer a missing product attribute

Predict the missing `Color` value for product `ProductID=870`.

In [ ]:
query_attribute = PredictiveQuery(
    'PREDICT products.Color FOR EACH products.ProductID',
    schema,
)

request = query_attribute.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the product with `ProductID=870`
2. `prediction`: the predicted color category
3. `probability_*` columns: the estimated likelihood of each category

Attribute inference can help enrich incomplete catalog records, but predicted values should be reviewed before being written back to a source system.

### Example 5: Predict customer churn

Predict the likelihood that customers `CustomerID=11176` and `CustomerID=11091` will place zero orders in the next 90 days, reusing Example 2's query with one addition. The `WHERE` condition does not filter which entities are scored, both `indices` are still predicted; it filters which rows qualify as labeled context examples, restricting the model's learned comparison to customers who themselves placed at least one order in the previous 90 days.

In [ ]:
query_churn = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID '
    'WHERE COUNT(sales_order_headers.*, -90, 0, days) > 0',
    schema,
)

request = query_churn.materialize(indices=[11176, 11091])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the customer represented by each `CustomerID`
2. `prediction`: whether the customer is predicted to place zero orders during the next 90 days
3. `probability_*` columns: the estimated likelihood of each outcome

This churn prediction can support retention or re-engagement workflows.

## Step 7. Close the client

Close the client when the notebook is finished to release its HTTP connection.

In [ ]:
client.close()